# F2 · Ridge with and without scaling: the same penalty is not the same model

**Type**: Compare · **Level**: ●●○ independent · **Time**: about 40 minutes · **core path ★**

## Objective

Show that OLS is invariant to the scale of the features and Ridge is not, and measure **by how much** the same penalty
alpha shrinks features in different units, finding which alpha is needed to halve the effect of each one.

## Dataset

Synthetic: three "true" variables `z1, z2, z3` with standard deviation 1, each with the same effect on the target
(`y = 2·(z1 + z2 + z3) + noise`). The three features that the model sees are the same variables expressed in very different units:
`x1 = z1`, `x2 = 30·z2`, `x3 = 1000·z3`. The three features have exactly the same real importance.

## Educational purpose

See numerically why "scale the features before Ridge or Lasso" is a rule and not a suggestion. You will measure the
**contribution** of each feature (coefficient × standard deviation), which is the quantity comparable across different
units.

## Task

1. Train `lin_raw = LinearRegression()` on `X` and `lin_scaled = make_pipeline(StandardScaler(), LinearRegression())`. Assign to `lin_invariant` `True` if the predictions of the two models coincide (`np.allclose`). Save in `ols_contributions` the contribution of each feature of the raw model: `|coef| · std(feature)` (`Series` with index `x1, x2, x3`).
2. With `Ridge(alpha=200)` compute `ridge_raw_contributions` (raw coefficient × std) and `ridge_scaled_contributions` (the coefficient of the pipeline with `StandardScaler`, which is already per one standard deviation). Compute `raw_ratio = contribution of x1 / contribution of x3` for the first and `scaled_ratio` for the second.
3. On a grid `alphas = np.logspace(-1, 10, 111)` find, for each feature and for each version (raw and scaled), the **first alpha for which the contribution drops below half of that of OLS**. Save the table in `half_alpha` (rows `raw` and `scaled`, columns `x1, x2, x3`).
4. Choose in `answer` ("a", "b" or "c") the correct explanation of the result:
   - **a**: Ridge without scaling penalizes more the features with a large standard deviation;
   - **b**: without scaling, the same alpha shrinks a lot the features in small units (x1) and hardly at all those in large units (x3), even though they all have the same real effect;
   - **c**: scaling changes nothing, because Ridge is scale invariant like OLS.

## Setup

Run this cell without changing it.

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

rng = np.random.default_rng(0)
n = 200
Z = rng.normal(size=(n, 3))                            # three true variables, standard deviation 1
X = pd.DataFrame(Z * np.array([1.0, 30.0, 1000.0]), columns=["x1", "x2", "x3"])
y = 2 * Z.sum(axis=1) + rng.normal(size=n)             # same effect (2 per standard deviation) for all three
stds = X.std()
stds.round(1)

## Your work

In [ ]:
lin_raw = ...
lin_scaled = ...
lin_invariant = ...
ols_contributions = ...
ols_contributions.round(2)

In [ ]:
ridge_raw = Ridge(alpha=200).fit(X, y)
ridge_scaled = make_pipeline(StandardScaler(), Ridge(alpha=200)).fit(X, y)
ridge_raw_contributions = ...
ridge_scaled_contributions = ...
raw_ratio = ...
scaled_ratio = ...

In [ ]:
alphas = np.logspace(-1, 10, 111)
half_alpha = pd.DataFrame(index=["raw", "scaled"], columns=["x1", "x2", "x3"], dtype=float)
for a in alphas:
    ...
half_alpha

In [ ]:
answer = ...

## Check

Run the cell when you think you are done. If a check fails, the message tells you what is wrong. If it fails a second time, a hint appears under the message.

In [ ]:
with verify(hint='For every alpha train `Ridge(alpha=a)` on the raw data and `make_pipeline(StandardScaler(), Ridge(alpha=a))` on the scaled, compute the contributions (`np.abs(coef) * stds.to_numpy()` for the raw, `np.abs(coef)` for the scaled) and, for every column where `half_alpha` is still `NaN`, record `a` when the contribution drops below `ols_contributions / 2`.'):
    check(bool(lin_invariant), "OLS is scale invariant: same predictions with and without the scaler",
          "the predictions of LinearRegression should coincide with and without StandardScaler")
    check(np.allclose(ols_contributions, 2.0, atol=0.5),
          "in comparable units the three features have the same contribution (about 2)", "contribution = |coef| * std of each feature")
    check(raw_ratio < 0.65, "without scaling Ridge shrinks x1 much more than x3",
          "the contribution of x1 should be reduced to about half of that of x3")
    check(0.85 < scaled_ratio < 1.2, "with scaling the three features are shrunk equally",
          "after scaling the ratio between the contributions should be about 1")
    check(half_alpha.loc["raw", "x3"] / half_alpha.loc["raw", "x1"] > 1e5,
          "without scaling the alpha that halves x3 is over a hundred thousand times the one that halves x1",
          "without scaling the scale of alpha needed should change by many orders of magnitude from x1 to x3")
    check(half_alpha.loc["scaled"].max() / half_alpha.loc["scaled"].min() < 3,
          "with scaling alpha has the same effect on all the features", "with scaling the three alphas should be about equal")
    check(answer == "b", "correct explanation", "reread the half_alpha table")
done()

---
The commented solution is in `solutions/F_compare/F2_ridge_with_and_without_scaling.ipynb`. Open it after you pass the check, or after you have really tried.